# Bonus +10% — t-SNE / UMAP de respuestas reales del agente Sándwich Qbano

Notebook anexo del Taller 3 (Ruta Transversal A — opcional avanzado).

Lo que hace este notebook:

1. Levanta del Postgres todas las respuestas del agente (`role='assistant'`, `route IS NOT NULL`).
2. Las vectoriza con el mismo modelo de embeddings que usa el RAG (`paraphrase-multilingual-MiniLM-L12-v2`, 384d) para mantener el análisis dentro del mismo espacio semántico que la operación real.
3. Proyecta a 2D con **t-SNE** (preserva estructura local) y **UMAP** (preserva estructura global).
4. Calcula el silhouette score en el espacio original como métrica académica honesta.
5. Interpreta los clústeres con honestidad: lo que el plot muestra, no lo que esperábamos que mostrara.

Todo el código vive también en `scripts/run_tsne_analysis.py` para que sea ejecutable desde la línea de comandos. Este notebook es la versión "para sustentar en clase".

## 1. Setup — imports + conexión a Postgres

In [ ]:
import sys
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while ROOT.name != 'proyecto' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import psycopg
import plotly.express as px
from dotenv import load_dotenv
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
import umap

load_dotenv(dotenv_path=ROOT / '.env', override=True)

from src.checkpointer import get_db_url
from src.vector_store import get_embeddings

print('Setup OK — corpus path:', ROOT)

## 2. Levantamos los turnos del agente desde Postgres

Solo respuestas del asistente con `route` asignada. Los turnos del usuario no llevan `route` (la decisión de routing es del agente, no del usuario), por eso no los vectorizamos.

In [ ]:
with psycopg.connect(get_db_url()) as conn:
    rows = conn.execute("""
        SELECT content, route, thread_id
        FROM conversation_messages
        WHERE role = 'assistant' AND route IS NOT NULL AND length(trim(content)) > 0
        ORDER BY id
    """).fetchall()

texts   = [r[0] for r in rows]
routes  = [r[1] for r in rows]
threads = [r[2] for r in rows]

print(f'Total turnos: {len(texts)}')
print(f'Threads únicos: {len(set(threads))}')
print()
print('Distribución por ruta:')
for route, count in sorted(Counter(routes).items(), key=lambda kv: -kv[1]):
    print(f'  {route:40s} {count:3d}  ({count/len(routes):.1%})')

dominant = max(Counter(routes).items(), key=lambda kv: kv[1])
print()
print(f'⚠️  Corpus desbalanceado: "{dominant[0]}" representa el {dominant[1]/len(routes):.0%}.')
print('    Cualquier métrica global de clustering queda sesgada por esa dominancia.')

## 3. Vectorizamos con el mismo modelo que usa el RAG

Esto es importante: si usáramos otro embeddings model, los clústeres podrían reflejar el sesgo del modelo, no las decisiones del agente. Manteniendo `paraphrase-multilingual-MiniLM-L12-v2` aseguramos que el análisis es coherente con lo que el agente ve en producción.

In [ ]:
embeddings = get_embeddings()
X = np.asarray(embeddings.embed_documents(texts))
print(f'Matriz de embeddings: {X.shape}  (turnos × dimensiones)')

## 4. Silhouette score en el espacio original

Calculamos esta métrica **antes** de reducir dimensionalidad, en el espacio embedding completo (384d). Esto garantiza que la medida no dependa del algoritmo de reducción.

Calibración académica del silhouette:

| Rango | Lectura |
|-------|---------|
| > 0.70 | Estructura fuerte. |
| 0.50 – 0.70 | Estructura razonable. |
| 0.25 – 0.50 | Estructura débil pero defendible. |
| < 0.25 | Prácticamente sin estructura. |

In [ ]:
y = LabelEncoder().fit_transform(routes)
sil = silhouette_score(X, y, metric='cosine')
print(f'Silhouette score (cosine, 384d original) = {sil:.4f}')
print()
if sil > 0.5:
    veredicto = 'Estructura clara: los clústeres están bien separados.'
elif sil > 0.25:
    veredicto = 'Estructura razonable: clústeres distinguibles aunque con solapamientos.'
elif sil > 0.10:
    veredicto = 'Estructura débil: existe alguna separación pero con mucho solapamiento; no se presenta como evidencia fuerte.'
else:
    veredicto = 'Prácticamente sin estructura: los embeddings se mezclan en el espacio original.'
print('Lectura:', veredicto)

## 5. t-SNE — proyección 2D que preserva estructura local

Perplexity baja porque el dataset es chico. Pasamos cosine como métrica porque es la natural para embeddings semánticos.

In [ ]:
perplexity = min(12, max(5, len(X) // 4))
tsne_coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric='cosine',
    init='pca',
    learning_rate='auto',
    max_iter=1500,
    random_state=42,
).fit_transform(X)

fig_tsne = px.scatter(
    x=tsne_coords[:, 0],
    y=tsne_coords[:, 1],
    color=routes,
    hover_data={
        'preview': [t[:120].replace('\n', ' ') + ('…' if len(t) > 120 else '') for t in texts],
        'thread': threads,
    },
    title=f't-SNE 2D — {len(texts)} respuestas del agente coloreadas por ruta',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2', 'color': 'Ruta'},
    opacity=0.8,
)
fig_tsne.update_traces(marker=dict(size=11, line=dict(width=0.4, color='white')))
fig_tsne.update_layout(template='plotly_white', width=1100, height=650)
fig_tsne.show()

## 6. UMAP — proyección 2D que preserva estructura global

Mostrar ambas técnicas es deliberado. Si los clústeres se ven en las dos, la separación es robusta y no un artefacto de un algoritmo específico. Si solo se ven en una, hay que tomar la lectura con pinzas.

In [ ]:
umap_coords = umap.UMAP(
    n_components=2,
    n_neighbors=min(15, len(X) - 1),
    min_dist=0.1,
    metric='cosine',
    random_state=42,
).fit_transform(X)

fig_umap = px.scatter(
    x=umap_coords[:, 0],
    y=umap_coords[:, 1],
    color=routes,
    hover_data={
        'preview': [t[:120].replace('\n', ' ') + ('…' if len(t) > 120 else '') for t in texts],
        'thread': threads,
    },
    title=f'UMAP 2D — {len(texts)} respuestas del agente coloreadas por ruta',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'color': 'Ruta'},
    opacity=0.8,
)
fig_umap.update_traces(marker=dict(size=11, line=dict(width=0.4, color='white')))
fig_umap.update_layout(template='plotly_white', width=1100, height=650)
fig_umap.show()

## 7. Interpretación honesta del plot

> Nota metodológica: t-SNE asigna coordenadas arbitrarias y la orientación cambia entre runs. Por eso esta sección habla en términos de **agrupamiento relativo**, no de posiciones absolutas.

**`consultar_datos_contacto`** — Contraintuitivamente, **es la clase más dispersa**, no la más densa. Aunque todas estas respuestas comparten una frase introductoria, el contenido cambia mucho entre turnos (WhatsApp, redes sociales, cobertura por ciudad, horarios). El embedding captura el cuerpo de la respuesta más que la frase de molde.

**`buscar_catalogo_productos`** — Es la clase con **agrupamiento visual más claro**. Sus respuestas son casi siempre tablas de precios con formato similar; el embedding identifica ese patrón estructural y los acerca.

**`memory`, `conversation`, `consultar_informacion_corporativa`** — Con 4-6 ejemplos cada una no se puede afirmar que formen clústeres estables. El tamaño muestral no es suficiente para conclusión estadística.

**`solicitar_supervisor_humano`** — Un solo punto. La estadística es trivial; solo se confirma que el sistema lo registró correctamente.

## Lo que el plot enseña, dicho sin inflar

1. **El agente sí responde diferente según la ruta**, pero la diferencia no se traduce automáticamente en clústeres separados en el espacio semántico.
2. **La intuición inicial fue equivocada**: las respuestas determinísticas no se agrupan mejor por seguir molde; el contenido pesa más que la plantilla.
3. **El silhouette bajo es coherente con lo que se ve**. No es un fallo del agente: es una limitación del corpus (desbalanceado, sesgado por pruebas de desarrollo).

## Qué se podría hacer con más datos reales

Con un mes de operación real (decenas de miles de turnos diversos):

1. Aparecería un clúster propio para **conversaciones fallidas** (turnos que cayeron al fallback cortés).
2. Picos en `solicitar_supervisor_humano` se podrían correlacionar con días específicos → señal operativa accionable.
3. Sub-clústeres densos dentro de `consultar_informacion_corporativa` indicarían preguntas frecuentes mal resueltas que ameritan entrar al JSON estructurado.

## Archivos también generados por `scripts/run_tsne_analysis.py`

| Archivo | Para qué |
|---------|----------|
| `results/tsne_2d_static.png` | Imagen embebida en el informe PDF. |
| `results/tsne_2d_interactive.html` | Gráfico para abrir en navegador con hover. |
| `results/umap_2d_interactive.html` | Versión UMAP del mismo análisis. |
| `results/tsne_analysis.md` | Resumen escrito (este notebook es la versión interactiva). |